## Pandas: Combining Datasets

In [ ]:
#import and auxiliary functions
import pandas as pd
import numpy as np

def make_df(cols , rows ):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in rows] for c in cols}
    return pd.DataFrame(data, rows)

# example DataFrame
make_df('ABC', range(3))

#### Array Contatenation using concat

In [ ]:
x = [[1,2],
    [3,4]]
np.concatenate([x,x])  #default :  axis=0 => vertical/row direction

In [ ]:
np.concatenate([x,x], axis=1)  # axis=1 => horizonatal/column direction

#### DataFrame/Series Contatenation using concat()
pd.concat  (objs, axis=0, join='outer', join_axes=None, <br>
            ignore_index=False,keys=None, levels=None, <br>
            names=None, verify_integrity=False, copy=True)

In [ ]:
# Series
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([ser1, ser2])  #default :  axis=0 => vertical/row direction

In [ ]:
pd.concat([ser1, ser2],axis=1)  # axis=1 => horizonatal/column direction

In [ ]:
#DataFrames
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
print(df1,'\n'); print(df2,'\n')

#Concatination => Union of items (row/columns) based on the direction

# Union of rows
print(pd.concat([df1, df2]), '\n')  #default :  axis=0 => vertical/row direction ; join="outer" => includes all columns
# ; join="inner"  => includes common columns
# Union of columns
print(pd.concat([df1, df2], axis=1)) # axis=1 => horizonatal/column direction ; join="outer" => includes all rows
# ; join="inner"  => includes common  rows with common key values

In [ ]:
print(pd.concat([df1, df2], axis=1, join="inner")) #returns only the rows that have matching keys in both DataFrames.

In [ ]:
##Duplicated indices in the left and right dataframes
print(pd.concat([df1, df1]), '\n')

In [ ]:
#Prevent the result from including duplicate indexes
print(pd.concat([df1, df1], verify_integrity=True))  # ERROR!!

In [ ]:
# ignore_index=True => reorder index
print(pd.concat([df2, df1]), '\n')
print(pd.concat([df2, df1], ignore_index=True))  # replace the old indices by the reordered new indices

In [ ]:
# add keys  to indicate sources
df3 = df1
df4 = pd.concat([df1, df3], keys=['df1','df3'])  # added keys become the 1st level keys and the old keys become the 2nd level keys
print(df4)

In [ ]:
df4.index

In [ ]:
# union/intersection of the input columns
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])
print(df5,'\n'); print(df6,'\n')

#union: combine rows without check matching keys
print(pd.concat([df5, df6]),'\n')   #default :  axis=0 => vertical/row direction ; join="outer" => all common columns

#intersection: combine rows with check matching 
print(pd.concat([df5, df6],join='inner'))  # axis=0 => vertical/row direction ; Inner join - only common columns

## Concatenation using merge
![SQL Merge Types ](sql_merge.png)
### Merge Type:
By default, keys are common columns of both side dfs <br>
You can use  **on** , **left_on**  or **right_on** parameters to define the keys <br>

Rows from:
1. Inner (default): only rows with matching keys
2. Left: all rows from left df and right with matching keys
3. Right: all rows from right df and left with matching keys
4. Outer : all rows from both dfs <br>

Columns from the left and right dfs

In [ ]:
left =pd.DataFrame( {'key': ['K0','K1','K2','K3'],
                    'A':['A0','A1','A2','A3'],
                    'B':['B0','B1','B2','B3']})
right =pd.DataFrame({'key': ['K0','K1','K4','K5'],
                    'C':['C0','C1','C4','C5'],
                    'D':['D0','D1','D4','D5']})
print(left,'\n\n',right)


In [ ]:
print('Inner: \n',pd.merge(left, right, how='inner'),'\n')
print('Left: \n',pd.merge(left, right, how='left'),'\n')
print('Right: \n',pd.merge(left, right, how='right'),'\n')
print('Outer: \n',pd.merge(left, right, how='outer'),'\n')

## Row-wise consideration
#### Note: For inner join, the resulted table row numbers depend on:
#### both keys are unique => 1-to-1 =>  number of rows equal to min(df1, df2)
#### if one  key is unique and another key is not unique => 1-to-many or many-to-1 =>  number of rows equal to max(df1, df2) 
#### if both keys are not unique => many-to-many =>  number of rows  equal to the Combination of df1 & df2 

In [ ]:
#one to one join
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
print(df1,'\n'); print(df2,'\n')

In [ ]:
df1.merge(df2, left_on='employee', right_on='employee')

In [ ]:
df3 = pd.merge(df1, df2) # default how='inner'
print(df3,'\n')

In [ ]:
#many to one join
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
print(df3,'\n'); print(df4,'\n'); 
#additional column with the “supervisor” information,
# information repeated as required by the inputs
print(pd.merge(df3, df4))   # default how='inner'

In [ ]:
#many to many join
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux',
                               'spreadsheets', 'organization']})
print(df1,'\n'); print(df5,'\n')
print(pd.merge(df1, df5))
#group correspond to two skills, thus two rows per employee

In [ ]:
# specify the merge key
# optional if there is only one pair of common keys
print(df1,'\n'); print(df2,'\n'); print(pd.merge(df1, df2, on='employee'))

In [ ]:
pd.concat([df1,df2],axis=1) #undesired

In [ ]:
# different keys for different datasets
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})
print(df1,'\n'); print(df3,'\n');
print(pd.merge(df1, df3, left_on="employee", right_on="name"))

In [ ]:
pd.merge(df1, df3, 
         left_on="employee", right_on="name").drop('name', axis=1)



In [ ]:
#index merge: employee as the row index of the two dfs
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')
print(df1a,'\n'); print(df2a,'\n')

In [ ]:
#then merge using indices
print(pd.merge(df1a, df2a, left_index=True, right_index=True))

In [ ]:
# join(): merge using indices by default
print(df1a.join(df2a))

In [ ]:
df1a

In [ ]:
# mixed of index and column name as keys
print(pd.merge(df1a, df3, left_index=True, right_on='name'))

In [ ]:
#example
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']},
                   columns=['name', 'food'])
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']},
                   columns=['name', 'drink'])
print(df6,'\n'); print(df7,'\n'); 

In [ ]:
print(pd.merge(df6, df7))
#equivalent
print(pd.merge(df6, df7, how='inner'))

In [ ]:
#how argument
print(pd.merge(df6, df7, how='outer'),'\n')

print(pd.merge(df6, df7, how='left'))

#### Overlapping column names
both left and right dfs have same column names => add suffixes to the column names in the resulted dfs

In [ ]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [3, 1, 4, 2]})
print(df8,'\n'); print(df9,'\n')

In [ ]:
print(pd.merge(df8, df9, on="name"),'\n')  # add suffuxes by pandas
print(pd.merge(df8, df9, on="name", suffixes=["_L", "_R"])) # add suffuxes by users

##### Exercise 1
Append the information from an existing Pandas series to an existing DataFrame, reindex the row index, and display the combined data.

In [ ]:
import pandas as pd
student_data1  = pd.DataFrame({
        'student_id': ['S1', 'S2', 'S3', 'S4', 'S5'],
         'name': ['Danniella Fenton', 'Ryder Storey', 'Bryce Jensen', 'Ed Bernal', 'Kwame Morin'], 
        'marks': [200, 210, 190, 222, 199]})

record = pd.Series(['S6', 'Scarlette Fisher', 205], index=['student_id', 'name', 'marks'])

#### your answer here


In [ ]:
df = pd.DataFrame(record)
df

In [ ]:
s2=pd.DataFrame(student_data2).unstack(level=0)
s4 =pd.DataFrame(s2.unstack())
combined_data =  pd.concat([student_data1, s4], ignore_index=True)

In [ ]:
combined_data

##### Exercise 2
(1) Join the two given dataframes 'student_data1' and 'student_data2' along columns and store it as 'df1'.

(2) For 'student_data1', set 'name' as the row index and save it as 'index_data1'.

(3) Merge the dataframes 'index_data1' and 'student_data2' using student names, keep the union of all student data, remove any id relevant columns and store it as 'df2'.

(4) Reindex 'df2' starting from 0 and save it as 'df3'.

(5) Impute/Replace missing values in 'df3' with 0 and save it as 'df4'.

In [ ]:
import pandas as pd
student_data1 = pd.DataFrame({
        'studentid': ['S1', 'S2', 'S3', 'S4', 'S5'],
         'name': ['Dante Morse', 'Ryder Storey', 'Bryce Jensen', 'Ed Bernal', 'Kwame Morin'], 
        'history_marks': [200, 210, 190, 222, 199]})

student_data2 = pd.DataFrame({
        'student_id': ['S6', 'S4', 'S1', 'S9', 'S10'],
        'name': ['Scarlette Fisher', 'Ed Bernal', 'Dante Morse', 'Kaiser William', 'Madeeha Preston'], 
        'math_marks': [201, 200, 198, 219, 201]})

print("Original DataFrames:")
print(student_data1)
print(student_data2)

In [ ]:
## (1)
#### your answer here


In [ ]:
## (2)
#### your answer here


In [ ]:
## (3)
#### your answer here


In [ ]:
## (4)
#### your answer here


In [ ]:
## (5)
#### your answer here


### GroupBy: Conditional Aggregation

In [ ]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(1)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6),
                   'random': rng.random(6)}, columns=['key', 'data', 'random'])
df

In [ ]:
# DataFrameGroupBy object: group data by the desired key column
key_group = df.groupby('key')

In [ ]:
print(key_group.sum(),'\n')

In [ ]:
print(key_group['random'].sum())

In [ ]:
print(key_group.min())

In [ ]:
# iteration over groups
for (key, group) in key_group:
    print(key,'\n',group)
    print("{} shape={} \n".format(key, group.shape))

In [ ]:
# describe()
df.groupby('key')['random'].describe()

#### Aggregate, filter, transform, apply

In [ ]:
#agg() take a list of function names or lamda functions 
df.groupby('key').aggregate(['min', 'median', 'max'])

In [ ]:
#use dictionary rather than list for mapping specific functions to specific columns
df.groupby('key').aggregate({'data': 'min',
                             'random': ['min','max']})

In [ ]:
#filtering
#keep groups that meet certain criteria
def filter_func(x): # x is the group object created by the groupby() method
    return x['random'].min() < 0.3  #return the groups with group minimum less than 0.3
    
print(df, '\n'); 
print(df.groupby('key').filter(filter_func))  
# It filters entire groups from a DataFrame based on whether the group as a whole meets a condition.

In [ ]:
# filter() on DataFrames/Series
# filters columns based on their names, using specific criteria.
df.filter(like = 'rand')

In [ ]:
df

In [ ]:
# transformation;  use an aggregated value from a group(e.g. group mean) to operte on every row in that group.
# example:convert the items on every row by subtracting the group-wise mean
df.groupby('key').transform(lambda x: x - x.mean())

In [ ]:
# apply a function to the group results
# apply() is a method that applies a function along a specific axis (rows or columns) of a DataFrame or to each element of a Series.
# Think of it as a way to "loop" over your data in a more efficient and readable way, without writing explicit for loops.


def norm_by_data2(x): 
    #must take a dataframe as its first argument and return a DataFrame, Series or scalar.
    x['random'] /= x['data'].sum()
    return x

print(df,'\n')
g1 = df.groupby(df['key'], group_keys=False)
g2 = df.groupby(df['key'], group_keys=True)
print(g1[['data','random']].apply(norm_by_data2)) # no group index displayed
print(g2[['data','random']].apply(norm_by_data2)) # group index displayed

#### Specification of the split key

In [ ]:
#group data by mapping
df2 = df.set_index('key')  # convert the 'key' column as index column
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'} # rename/regroup the index column items 
print(df2,'\n'); print(df2.groupby(mapping).sum())

In [ ]:
#group data by function
df3 = df2.groupby(str.lower)
print(df3.mean())

In [ ]:
#group data by multi-index
df3 = df2.groupby([str.lower, mapping]).mean()
df3

In [ ]:
df3.index

##### Exercise 3
(1) Split the following dataframe into groups based on school code and class. Obtain mean, min, and max value of age for each school.

(2) Consider only the schools that have student age <= 13. Split the dataframe into groups based on school code, and obtain the average of BMI for each school. BMI: body mass index is defined as weight/(height/100)^2.

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
student_data = pd.DataFrame({
    'school_code': ['s001','s002','s003','s001','s002','s004'],
    'class': ['V', 'V', 'VI', 'VI', 'V', 'VI'],
    'name': ['Alberto Franco','Gino Mcneill','Ryan Parkes', 'Eesha Hinton', 'Gino Mcneill', 'David Parkes'],
    'date_Of_Birth ': ['15/05/2002','17/05/2002','16/02/1999','25/09/1998','11/05/2002','15/09/1997'],
    'age': [12, 12, 13, 13, 14, 12],
    'height': [173, 192, 186, 167, 151, 159],
    'weight': [70, 85, 93, 65, 50, 60],
    'address': ['street1', 'street2', 'street3', 'street1', 'street2', 'street4']},
    index=['S1', 'S2', 'S3', 'S4', 'S5', 'S6'])
print("Original DataFrame:")
print(student_data)

In [ ]:
## (1) 
#### your answer here


In [ ]:
## (2) 
#### your answer here


### Pivot Table

In [ ]:
# example dataset
import numpy as np
import pandas as pd
import seaborn as sns
titanic = sns.load_dataset('titanic')
titanic.head(6)

In [ ]:
# group by class and gender
# select survival, apply a mean aggregate

titanic.groupby(['sex', 'class'])['survived'].aggregate('mean')

In [ ]:
# unstack the hierarchical index
titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack()

In [ ]:
''' pivot table alternative
df.pivot_table(values=None, index=None, columns=None, aggfunc='mean', fill_value=None, margins=False)
• values: The numerical column(s) you want to aggregate.
• index: The column(s) to group by. These become the new row labels.
• columns: The column(s) whose unique categories become the new column headers.
• aggfunc: The math operation to perform (e.g., 'mean', 'sum', 'count', 'max'). Defaults to 'mean'.
• fill_value: Replaces resulting missing values (NaN) with a custom fallback (e.g., 0).
• margins: Set to True to add a "Total" column showing grand totals. '''

titanic.pivot_table('survived', index='sex', columns='class')  # default aggfunc='mean'

In [ ]:
#multilevel pivot tables
#a third dimension, as an example
age = pd.cut(titanic['age'], bins= [0, 18, 60, 100],labels=["child", "adult", "senior"])
titanic.pivot_table('survived', ['sex', age], 'class')

In [ ]:
# multilevel at columns
fare_category = pd.qcut(titanic['fare'],  q=2) # change the 'fare' to categorial data
# number of quantiles = 2 => fare_category has only two categories
titanic.pivot_table('survived', ['sex', age], [fare_category, 'class'])

In [ ]:
#check on the quantiles
titanic['fare'].quantile(0.5)

In [ ]:
#aggfunc: controls what type of aggregation is applied
titanic.pivot_table(index='sex', columns='class',
                    aggfunc={'survived':sum, 'fare':'mean'})

In [ ]:
#margins
titanic.pivot_table('survived', index='sex', columns='class', 
                    margins=True)  # Add row/column margins (subtotals)

##### Exercise 4

For the titanic data, find survival rate by gender, age of the different categories (0,18) (18,40) (40,65) (65,80) of various classes.

In [ ]:
#### your answer here
